In [1]:
import pandas as pd

In [2]:
DATA_PATH = "../data/raw/en_fr.parquet"

In [3]:
df = pd.read_parquet(DATA_PATH)
df.head()

,id,translation
0,0,"{'en': 'The Wanderer', 'fr': 'Le grand Meaulnes'}"
1,1,"{'en': 'Alain-Fournier', 'fr': 'Alain-Fournier'}"
2,2,"{'en': 'First Part', 'fr': 'PREMIÈRE PARTIE'}"
3,3,"{'en': 'I', 'fr': 'CHAPITRE PREMIER'}"
4,4,"{'en': 'THE BOARDER', 'fr': 'LE PENSIONNAIRE'}"


In [4]:
df["en"] = df["translation"].apply(lambda x: x["en"])
df["fr"] = df["translation"].apply(lambda x: x["fr"])

df = df[["en", "fr"]]
df.head()

,en,fr
0,The Wanderer,Le grand Meaulnes
1,Alain-Fournier,Alain-Fournier
2,First Part,PREMIÈRE PARTIE
3,I,CHAPITRE PREMIER
4,THE BOARDER,LE PENSIONNAIRE


In [5]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

In [6]:
train_df.to_parquet("../data/raw/train.parquet", index=False)
val_df.to_parquet("../data/raw/val.parquet", index=False)
test_df.to_parquet("../data/raw/test.parquet", index=False)

In [7]:
train_df.shape, val_df.shape, test_df.shape

((101668, 2), (12708, 2), (12709, 2))

In [8]:
train_df = pd.read_parquet("../data/raw/train.parquet")
train_df.head()

,en,fr
0,"The truth is, he ought to have been trusted wi...","La vérité est que j'eusse dû lui confier tout,..."
1,"To absorb it, we would need to fill containers...","Pour l'absorber, il eût fallu remplir des réci..."
2,"""Ah, that’s quite another thing; but promise m...",-- Alors c'est autre chose; mais promettez-moi...
3,"""Now for the lad; look sharp.""","Au mioche maintenant, dépechons!"
4,'I would give my life a thousand times to know...,– Je donnerais mille fois ma vie pour savoir c...


In [9]:
from machine_translation.tokenization import load_tokenizer
from machine_translation.config import load_tokenizer_config, load_data_config

In [10]:
TOKENIZER_PATH = "../artifacts/tokenizer/tokenizer.json"
TOKENIZER_CONFIG_PATH = "../configs/tokenizers/shared_unigram.yaml"
DATA_CONFIG_PATH = "../configs/data/default.yaml"
tokenizer_config = load_tokenizer_config(TOKENIZER_CONFIG_PATH)
data_config = load_data_config(DATA_CONFIG_PATH)

In [11]:
tokenizer = load_tokenizer(TOKENIZER_PATH, tokenizer_config.special_tokens)
tokenizer

Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[{"id":0, "content":"<pad>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":1, "content":"<unk>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":2, "content":"<bos>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":3, "content":"<eos>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}], normalizer=NFKC(), pre_tokenizer=Metaspace(replacement="▁", prepend_scheme=always, split=True), post_processor=TemplateProcessing(single=[SpecialToken(id="<bos>", type_id=0), Sequence(id=A, type_id=0), SpecialToken(id="<eos>", type_id=0)], pair=[Sequence(id=A, type_id=0), Sequence(id=B, type_id=1)], special_tokens={"<bos>":SpecialToken(id="<bos>", ids=[2], tokens=["<bos>"]), "<eos>":SpecialToken(id="<eos>", ids=[3], tokens=["<eos>"])}), decoder=Metaspac

In [12]:
sample = "Before we start, let's make sure we have the right tools for the job."

encoded_sample = tokenizer.encode(sample)
print("Encoded sample:", encoded_sample)
print("Encoded sample:", encoded_sample.ids)
print("Decoded sample:", tokenizer.decode(encoded_sample.ids))
print("Tokens:", encoded_sample.tokens)

Encoded sample: Encoding(num_tokens=21, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])
Encoded sample: [2, 6310, 127, 3328, 4, 598, 115, 293, 1026, 127, 72, 6, 605, 6457, 49, 6, 1503, 120, 157, 5, 3]
Decoded sample: Before we start, let's make sure we have the right tools for the job.
Tokens: ['<bos>', '▁Before', '▁we', '▁start', ',', '▁let', "'s", '▁make', '▁sure', '▁we', '▁have', '▁the', '▁right', '▁tools', '▁for', '▁the', '▁j', 'o', 'b', '.', '<eos>']


In [13]:
from machine_translation import get_data_loaders

In [17]:
train_loader, validation_loader, test_loader = get_data_loaders(
    tokenizer=tokenizer,
    data_config=data_config,
    tokenizer_config=tokenizer_config,
)

In [19]:
len(train_loader), len(validation_loader), len(test_loader)

In [ ]:
for batch in train_loader:
    print(batch["source_ids"])
    print(batch["target_ids"])
    print(batch["source_padding_mask"])
    print(batch["target_padding_mask"])
    print("Source input IDs shape:", batch["source_ids"].shape)
    print("Target input IDs shape:", batch["target_ids"].shape)
    print("Source padding mask shape:", batch["source_padding_mask"].shape)
    print("Target padding mask shape:", batch["target_padding_mask"].shape)
    break